In [ ]:
%cd ..
%load_ext autoreload
%autoreload 2

/home/dongmin/userdata/open-score-string-quartets


/home/dongmin/.local/share/virtualenvs/open-score-string-quartets-wd2Cnojv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import sys
import subprocess
import warnings
from typing import Union, Any, Optional
import shutil
from pathlib import Path
from collections import Counter, defaultdict
from operator import itemgetter, contains, eq # contains(A, B) == (B in A), eq(A, B) == (A == B)
from itertools import groupby
from tempfile import NamedTemporaryFile, TemporaryDirectory

import math
import random

import json
import csv
import strictyaml as syaml

from tqdm import tqdm
import matplotlib.pyplot as plt

import cv2
import numpy as np
import partitura as pt

import xml.etree.ElementTree as ET
from lmx_utils import delinearize_lmx, render_xml
from lmx_utils.Linearizer import Linearizer, SQLinearizer
from lmx_utils.Delinearizer import Delinearizer
from lmx_utils.symbolic.MxlFile import MxlFile
from lmx_utils.symbolic.part_to_score import string_quartet_parts_to_score

dformat = lambda d: json.dumps(d, indent=2)
dprint = lambda d: print(dformat(d))

In [3]:
PathLike = Union[Path, str]

In [4]:
root_dir = Path(os.getcwd())
data_dir = root_dir / 'data'
score_dir = root_dir / 'scores'

In [5]:
data_dir, score_dir

(PosixPath('/home/dongmin/userdata/open-score-string-quartets/data'),
 PosixPath('/home/dongmin/userdata/open-score-string-quartets/scores'))

## Load metadata

In [6]:
with open(data_dir / 'scores.yaml') as f:
  score_metadata = syaml.load(f.read())

score_metadata = score_metadata.data

list(score_metadata.keys())[0]

'7313978'

In [8]:
score_metadata['7313978']

{'path': 'Andrée,_Elfrida/String_Quartet_in_A_major',
 'name': 'String Quartet in A major',
 'link': 'https://musescore.com/openscore-string-quartets/scores/7313978',
 'imslp': '#415720',
 'set_id': '5108429'}

In [7]:
with open(data_dir / 'flattened_metadata_2024-10-28-00:50:23.csv', 'r') as f:
  flattened_metadata = list(csv.reader(f))[1:]
  
flattened_metadata[0]

['sq7313978', '0001', '9605', '495', '5']

## exclude part scores

In [16]:
dict_by_path = defaultdict(list)
for m_id, infos in score_metadata.items():
  dict_by_path[infos['path']].append(m_id)

dict_by_path = dict(dict_by_path)

for path, indices in dict_by_path.items():
  if len(indices) > 1:
    print(path, indices)

Mayer,_Emilie/String_Quartet_in_G_minor,_Op.14 ['7236909', '7070319', '7082029', '7095930']
Mozart,_Wolfgang_Amadeus/String_Quartet_No.18_in_A_major,_K.464_(Op._10,_No._5) ['7093885', '7070781', '7078259', '7075297']


In [18]:
for m_id, infos in score_metadata.items():
  flattened_images = sorted(list(
    (score_dir / infos['path'] / 'images' / 'flattened').glob('*.png')
  ))
  
  for i_p in flattened_images:
    i_p.unlink()

In [20]:
flt_lens = []
for m_id, infos in score_metadata.items():
  flt_lens.append(len(list(
    (score_dir / infos['path'] / 'images' / 'flattened').glob('*.png')
  )))

any(flt_lens)

False

## verify flattened

In [9]:
flattened_dict = defaultdict(list)

for m_id, page_idx, width, height, n_staff in flattened_metadata:
  m_id = m_id.replace('sq', '')
  flattened_dict[m_id].append((m_id, page_idx, width, height, n_staff))

flattened_dict = dict(flattened_dict)

In [17]:
for m_id, infos in score_metadata.items():
  num_pages = len(list((score_dir / infos['path'] / 'images' / 'original').glob('*.png')))
  num_flattened = len(flattened_dict.get(m_id, []))
  if num_flattened != num_pages:
    print(m_id, num_pages, num_flattened, infos['path'])

7302602 41 18 Brahms,_Johannes/String_Quartet_No.2,_Op.51_No.2
7236909 23 8 Mayer,_Emilie/String_Quartet_in_G_minor,_Op.14
7070319 23 0 Mayer,_Emilie/String_Quartet_in_G_minor,_Op.14
7082029 23 0 Mayer,_Emilie/String_Quartet_in_G_minor,_Op.14
7095930 23 0 Mayer,_Emilie/String_Quartet_in_G_minor,_Op.14
7093885 20 8 Mozart,_Wolfgang_Amadeus/String_Quartet_No.18_in_A_major,_K.464_(Op._10,_No._5)
7070781 20 0 Mozart,_Wolfgang_Amadeus/String_Quartet_No.18_in_A_major,_K.464_(Op._10,_No._5)
7078259 20 0 Mozart,_Wolfgang_Amadeus/String_Quartet_No.18_in_A_major,_K.464_(Op._10,_No._5)
7075297 20 0 Mozart,_Wolfgang_Amadeus/String_Quartet_No.18_in_A_major,_K.464_(Op._10,_No._5)


## insert page infos

In [9]:
for m_id, page_idx, width, height, n_staff in flattened_metadata:
  m_id = m_id.replace('sq', '')
  if not score_metadata[m_id].get('pages', None):
    score_metadata[m_id]['pages'] = []
  
  score_metadata[m_id]['pages'].append({
    'page_index': page_idx,
    'n_staff': int(n_staff)
  })

In [10]:
score_metadata['7313978']

{'path': 'Andrée,_Elfrida/String_Quartet_in_A_major',
 'name': 'String Quartet in A major',
 'link': 'https://musescore.com/openscore-string-quartets/scores/7313978',
 'imslp': '#415720',
 'set_id': '5108429',
 'pages': [{'page_index': '0001', 'n_staff': 5},
  {'page_index': '0002', 'n_staff': 5},
  {'page_index': '0003', 'n_staff': 5},
  {'page_index': '0004', 'n_staff': 5},
  {'page_index': '0005', 'n_staff': 5},
  {'page_index': '0006', 'n_staff': 5},
  {'page_index': '0007', 'n_staff': 5},
  {'page_index': '0008', 'n_staff': 5},
  {'page_index': '0009', 'n_staff': 5},
  {'page_index': '0010', 'n_staff': 5},
  {'page_index': '0011', 'n_staff': 5},
  {'page_index': '0012', 'n_staff': 5},
  {'page_index': '0013', 'n_staff': 5},
  {'page_index': '0014', 'n_staff': 5},
  {'page_index': '0015', 'n_staff': 5},
  {'page_index': '0016', 'n_staff': 5},
  {'page_index': '0017', 'n_staff': 6},
  {'page_index': '0018', 'n_staff': 5},
  {'page_index': '0019', 'n_staff': 5},
  {'page_index': '002

In [11]:
with open(data_dir / 'scores_w_n_staff.yaml', 'w') as f:
  f.write(syaml.as_document(score_metadata).as_yaml())

In [12]:
with open(data_dir / 'scores_w_n_staff.yaml', 'r') as f:
  score_metadata = syaml.load(f.read()).data